<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/AML_merge_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Directories
# ------------------------------------------------------------

PATIENT_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/Clinical Patient/survival_model_ready"
)

SAMPLE_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/Clinical Sample/survival_model_ready"
)

MERGED_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/combined_survival_model_ready"
)

MERGED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)




In [ ]:
 # ------------------------------------------------------------
# 2. Merge function
# ------------------------------------------------------------

def merge_patient_and_sample(
    patient_path,
    sample_path,
    output_path
):
    patient_df = pd.read_csv(patient_path)
    sample_df = pd.read_csv(sample_path)

    print(f"\nProcessing: {output_path.name}")

    # --------------------------------------------------------
    # Check required columns
    # --------------------------------------------------------

    required_patient_cols = [
        "PATIENT_ID",
        "OS_DAYS",
        "OS_EVENT"
    ]

    required_sample_cols = [
        "PATIENT_ID",
        "SAMPLE_ID"
    ]

    missing_patient_cols = [
        col for col in required_patient_cols
        if col not in patient_df.columns
    ]

    missing_sample_cols = [
        col for col in required_sample_cols
        if col not in sample_df.columns
    ]

    if missing_patient_cols:
        raise KeyError(
            f"Patient dataset is missing columns: "
            f"{missing_patient_cols}"
        )

    if missing_sample_cols:
        raise KeyError(
            f"Sample dataset is missing columns: "
            f"{missing_sample_cols}"
        )

    # --------------------------------------------------------
    # Clean PATIENT_ID type and whitespace
    # --------------------------------------------------------

    patient_df["PATIENT_ID"] = (
        patient_df["PATIENT_ID"]
        .astype(str)
        .str.strip()
    )

    sample_df["PATIENT_ID"] = (
        sample_df["PATIENT_ID"]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # Check missing and duplicated patient IDs
    # --------------------------------------------------------

    if patient_df["PATIENT_ID"].isna().any():
        raise ValueError(
            "Patient dataset contains missing PATIENT_ID values."
        )

    if sample_df["PATIENT_ID"].isna().any():
        raise ValueError(
            "Sample dataset contains missing PATIENT_ID values."
        )

    if patient_df["PATIENT_ID"].duplicated().any():
        duplicated_ids = patient_df.loc[
            patient_df["PATIENT_ID"].duplicated(keep=False),
            "PATIENT_ID"
        ].unique()

        raise ValueError(
            "Patient dataset contains duplicate PATIENT_ID values: "
            f"{duplicated_ids[:10]}"
        )

    if sample_df["PATIENT_ID"].duplicated().any():
        duplicated_ids = sample_df.loc[
            sample_df["PATIENT_ID"].duplicated(keep=False),
            "PATIENT_ID"
        ].unique()

        raise ValueError(
            "Sample dataset contains duplicate PATIENT_ID values: "
            f"{duplicated_ids[:10]}"
        )

    # --------------------------------------------------------
    # Check overlapping column names
    # --------------------------------------------------------

    overlapping_cols = sorted(
        (
            set(patient_df.columns)
            & set(sample_df.columns)
        )
        - {"PATIENT_ID"}
    )

    if overlapping_cols:
        raise ValueError(
            "The two datasets contain overlapping columns other "
            f"than PATIENT_ID: {overlapping_cols}. "
            "Rename or remove duplicate columns before merging."
        )

    # --------------------------------------------------------
    # Check patient matching before merge
    # --------------------------------------------------------

    patient_ids = set(patient_df["PATIENT_ID"])
    sample_ids = set(sample_df["PATIENT_ID"])

    only_in_patient = sorted(
        patient_ids - sample_ids
    )

    only_in_sample = sorted(
        sample_ids - patient_ids
    )

    print(
        f"Patient rows: {len(patient_df)}"
    )

    print(
        f"Sample rows: {len(sample_df)}"
    )

    print(
        f"Patients only in patient dataset: "
        f"{len(only_in_patient)}"
    )

    print(
        f"Patients only in sample dataset: "
        f"{len(only_in_sample)}"
    )

    if only_in_patient:
        print(
            "Example IDs only in patient dataset:",
            only_in_patient[:10]
        )

    if only_in_sample:
        print(
            "Example IDs only in sample dataset:",
            only_in_sample[:10]
        )

    # --------------------------------------------------------
    # Perform one-to-one inner merge
    # --------------------------------------------------------

    merged_df = patient_df.merge(
        sample_df,
        on="PATIENT_ID",
        how="inner",
        validate="one_to_one"
    )

    print(
        f"Matched rows after inner merge: "
        f"{len(merged_df)}"
    )

    # --------------------------------------------------------
    # Validate survival outcomes
    # --------------------------------------------------------

    if merged_df["OS_DAYS"].isna().any():
        raise ValueError(
            "Missing OS_DAYS values found after merging."
        )

    if merged_df["OS_EVENT"].isna().any():
        raise ValueError(
            "Missing OS_EVENT values found after merging."
        )

    if (merged_df["OS_DAYS"] < 0).any():
        raise ValueError(
            "Negative OS_DAYS values found."
        )

    invalid_event_values = set(
        merged_df["OS_EVENT"].dropna().unique()
    ) - {0, 1}

    if invalid_event_values:
        raise ValueError(
            "OS_EVENT contains values other than 0 and 1: "
            f"{invalid_event_values}"
        )

    # --------------------------------------------------------
    # Check for missing or infinite predictor values
    # --------------------------------------------------------

    non_predictor_cols = [
        "PATIENT_ID",
        "SAMPLE_ID",
        "OS_DAYS",
        "OS_EVENT"
    ]

    predictor_df = merged_df.drop(
        columns=non_predictor_cols,
        errors="ignore"
    )

    if predictor_df.isna().any().any():
        missing_cols = predictor_df.columns[
            predictor_df.isna().any()
        ].tolist()

        raise ValueError(
            "Missing values found in merged predictors: "
            f"{missing_cols}"
        )

    numeric_predictors = predictor_df.select_dtypes(
        include=[np.number]
    )

    if np.isinf(
        numeric_predictors.to_numpy()
    ).any():
        raise ValueError(
            "Infinite values found in merged predictors."
        )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    merged_df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved to: {output_path}"
    )

    return merged_df



In [ ]:
# ------------------------------------------------------------
# 3. Merge train, validation and test
# ------------------------------------------------------------

train_combined = merge_patient_and_sample(
    PATIENT_DIR
    / "train_survival_model_ready.csv",

    SAMPLE_DIR
    / "sample_train_survival_model_ready.csv",

    MERGED_OUTPUT_DIR
    / "combined_train_survival_model_ready.csv"
)


val_combined = merge_patient_and_sample(
    PATIENT_DIR
    / "validation_survival_model_ready.csv",

    SAMPLE_DIR
    / "sample_validation_survival_model_ready.csv",

    MERGED_OUTPUT_DIR
    / "combined_validation_survival_model_ready.csv"
)


test_combined = merge_patient_and_sample(
    PATIENT_DIR
    / "test_survival_model_ready.csv",

    SAMPLE_DIR
    / "sample_test_survival_model_ready.csv",

    MERGED_OUTPUT_DIR
    / "combined_test_survival_model_ready.csv"
)



Processing: combined_train_survival_model_ready.csv
Patient rows: 620
Sample rows: 620
Patients only in patient dataset: 0
Patients only in sample dataset: 0
Matched rows after inner merge: 620
Saved to: /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/combined_survival_model_ready/combined_train_survival_model_ready.csv

Processing: combined_validation_survival_model_ready.csv
Patient rows: 177
Sample rows: 177
Patients only in patient dataset: 0
Patients only in sample dataset: 0
Matched rows after inner merge: 177
Saved to: /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/combined_survival_model_ready/combined_validation_survival_model_ready.csv

Processing: combined_test_survival_model_ready.csv
Patient rows: 89
Sample rows: 89
Patients only in patient dataset: 0
Patients only in sample dataset: 0
Matched rows after inner merge: 89
Saved to: /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/combined_survival_model_ready/combined_test_surviv

In [ ]:
# ------------------------------------------------------------
# 4. Check final shapes and columns
# ------------------------------------------------------------

print("\nFinal combined shapes:")

print(
    "Training:",
    train_combined.shape
)

print(
    "Validation:",
    val_combined.shape
)

print(
    "Test:",
    test_combined.shape
)


# Predictor columns should be identical across all splits.
train_cols = train_combined.columns.tolist()
val_cols = val_combined.columns.tolist()
test_cols = test_combined.columns.tolist()

if train_cols != val_cols:
    raise ValueError(
        "Training and validation columns differ."
    )

if train_cols != test_cols:
    raise ValueError(
        "Training and test columns differ."
    )

print(
    "\nAll three combined datasets have identical "
    "column names and column order."
)


Final combined shapes:
Training: (620, 68)
Validation: (177, 68)
Test: (89, 68)

All three combined datasets have identical column names and column order.
